A key assumption of the composite likelihood framework employed in this project is that within each bin,
observed statistics approximately follow the multivariate normal distribution. Here we use simulated and
empirical data to help justify this assumption.

In [3]:
import demes
import h2py
import msprime
import numpy as np

In [31]:
# Simulated analysis.

# Define parameters
r = 5e-8
u = 1.3e-8
n_chroms = 100
L = int(1e6)
# w_size = int(1e6)
# windows = [[i*w_size, (i+1)*w_size] for i in range(int(L/w_size))]
r_bins = np.logspace(-6, -2, 17)

# Load a (marginalized) version of the full model topography [todo]
graph = demes.load("data/graph.yaml")
demog = msprime.Demography.from_demes(graph)

pop_file = "data/pop_file"
samples = {"Yoruba1": 1, "Altai": 1}
sampled_demes = ["Yoruba1", "Altai"]


def run_simulation(fname):
    ts = msprime.sim_ancestry(
        samples,
        demography=demog,
        sequence_length=L,
        recombination_rate=r,
    )
    ts = msprime.sim_mutations(ts, rate=u)
    with open(fname, "w+") as f:
        ts.write_vcf(f, position_transform = lambda x: 1 + np.array(x),
                     individual_names=sampled_demes)

    sums = h2py.parsing.compute_h2_stats(
        vcf_file=fname,
        pop_file=pop_file,
        interval=[0, L],
        r=r,
        r_bins=r_bins,
        report=False,
    )
    print(h2py.utils.timestamp(), f"Parsed {fname}")
    return sums


run_simulation("temp_data/x")

[26-08-10 13:39:46] Parsed temp_data/x


{'pops': ['Yoruba1', 'Altai'],
 'stats': (['H2_0_0', 'H2_0_1', 'H2_1_1'], ['H_0_0', 'H_0_1', 'H_1_1']),
 'bins': [(1e-06, 1.778279410038923e-06),
  (1.778279410038923e-06, 3.162277660168379e-06),
  (3.162277660168379e-06, 5.623413251903491e-06),
  (5.623413251903491e-06, 1e-05),
  (1e-05, 1.778279410038923e-05),
  (1.778279410038923e-05, 3.1622776601683795e-05),
  (3.1622776601683795e-05, 5.623413251903491e-05),
  (5.623413251903491e-05, 0.0001),
  (0.0001, 0.00017782794100389227),
  (0.00017782794100389227, 0.00031622776601683794),
  (0.00031622776601683794, 0.0005623413251903491),
  (0.0005623413251903491, 0.001),
  (0.001, 0.0017782794100389228),
  (0.0017782794100389228, 0.0031622776601683794),
  (0.0031622776601683794, 0.005623413251903491),
  (0.005623413251903491, 0.01)],
 'sums': [array([27. , 38.5,  1. ]),
  array([42., 56.,  3.]),
  array([ 79.  , 117.25,   1.  ]),
  array([144., 227.,   0.]),
  array([211.  , 337.75,   3.  ]),
  array([386., 619.,   4.]),
  array([ 643.  , 1

In [9]:
# Empirical analysis.

In [10]:
# Generate a multi-panel supplementary figure.